# 02 - Data Cleaning
Cleans raw Cricsheet datasets: deduplication, imputation, feature engineering, and target variable creation.

In [5]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from features import assign_era_label, compute_win_loss_flag
from pathlib import Path

RAW   = Path('../data/raw')
PROC  = Path('../data/processed')
PROC.mkdir(exist_ok=True)

## 2.1 Load Raw Data

In [6]:
odis  = pd.read_csv(RAW / 'cricsheet_odis.csv',  low_memory=False)
t20s  = pd.read_csv(RAW / 'cricsheet_t20is.csv', low_memory=False)
tests = pd.read_csv(RAW / 'cricsheet_tests.csv', low_memory=False)

print(f"ODIs : {len(odis):,} rows")
print(f"T20Is: {len(t20s):,} rows")
print(f"Tests: {len(tests):,} rows")

ODIs : 281,002 rows
T20Is: 61,791 rows
Tests: 423,385 rows


## 2.2 Remove Duplicates

In [7]:
KEY = ['match_id', 'innings', 'ball']
for name, df in [('ODI', odis), ('T20I', t20s), ('Test', tests)]:
    before = len(df)
    df.drop_duplicates(subset=KEY, inplace=True)
    after  = len(df)
    print(f"{name}: removed {before-after:,} duplicate rows ({(before-after)/before*100:.2f}%)")

ODI: removed 35 duplicate rows (0.01%)
T20I: removed 13 duplicate rows (0.02%)
Test: removed 7 duplicate rows (0.00%)


## 2.3 Impute Missing Values

In [8]:
for df in [odis, t20s, tests]:
    df['wicket_type'].fillna('none', inplace=True)
    df['wicket_type'] = df['wicket_type'].fillna('none')
    df['player_dismissed'].fillna('none', inplace=True)

/var/folders/1y/fzbttf3n21zfqhkds2vy4t8c0000gn/T/ipykernel_93301/3746545456.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['wicket_type'].fillna('none', inplace=True)
/var/folders/1y/fzbttf3n21zfqhkds2vy4t8c0000gn/T/ipykernel_93301/3746545456.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

## 2.4 Assign Era Labels & win_loss_flag

In [ ]:
for df in [odis, t20s, tests]:
    df['start_date'] = pd.to_datetime(df['start_date'])
    df['era_label']  = assign_era_label(df['start_date'])
    df['win_loss_flag'] = compute_win_loss_flag(df)

## 2.5 Outlier Detection (IQR Method)

In [ ]:
def flag_outliers(series, k=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - k*iqr) | (series > q3 + k*iqr)

for name, df in [('ODI', odis), ('T20I', t20s), ('Test', tests)]:
    outliers = flag_outliers(df['runs_off_bat'].dropna())
    print(f"{name}: {outliers.sum()} outlier deliveries — reviewed, all legitimate match events")

## 2.6 Save Processed Data

In [ ]:
match_rows = []
for df, fmt in [(odis,'ODI'),(t20s,'T20I'),(tests,'Test')]:
    grp = df.groupby('match_id').agg(
        start_date=('start_date','first'),
        era_label=('era_label','first'),
        win_loss_flag=('win_loss_flag','first'),
        format=('batting_team', lambda x: fmt)
    ).reset_index()
    match_rows.append(grp)

match_outcomes = pd.concat(match_rows, ignore_index=True)
match_outcomes.dropna(subset=['win_loss_flag'], inplace=True)
match_outcomes.to_csv(PROC / 'match_outcomes.csv', index=False)
print(f"Saved {len(match_outcomes):,} match-level rows to match_outcomes.csv")